# 90 — Analyze SmolVLA depth-1 hybrid trees

Analyzes every **complete** tree currently logged under the fresh `v2-egl` experiment. The collection need not contain all 800 planned trees. Partial trees are reported and excluded from every outcome statistic.

In [ ]:
EXTRAS = 'analysis'
SETUP_ENV = False  # Analysis-only: do not require LeRobot/LIBERO/MuJoCo.
import urllib.request
exec(urllib.request.urlopen(
    'https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/'
    'pnp-vla/scripts/colab_bootstrap.py').read().decode())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pnp.smolvla_tree_analysis import (
    load_smolvla_tree_results, summarize_trees, uncertainty_ranking_summary)

tables = load_smolvla_tree_results()
trees, candidates = tables['trees'], tables['candidates']
complete = trees[trees.complete].copy()
display(tables['completion'])
print(f'Outcome analysis uses {len(complete)} complete trees and {len(candidates)} candidate rows.')

## Overall acquisition value and root-selection comparison

In [ ]:
overall = summarize_trees(trees)
by_strategy = summarize_trees(trees, 'selection_strategy')
by_source_outcome = summarize_trees(trees, 'source_success')
display(overall)
display(by_strategy)
display(by_source_outcome)

In [ ]:
plot = by_strategy.set_index('selection_strategy')
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
plot[['mixed_trees_pct', 'counterfactual_mixed_pct']].plot.bar(
    ax=axes[0], color=['#4C78A8', '#72B7B2'])
axes[0].set(title='Decision-sensitive trees', ylabel='Trees (%)', xlabel='Root selection')
plot[['oracle_gain_vs_stock_pp']].plot.bar(ax=axes[1], color='#F58518', legend=False)
axes[1].axhline(0, color='black', lw=1)
axes[1].set(title='Any-branch oracle gain', ylabel='Gain over stock (pp)', xlabel='Root selection')
plot[['fresh_failure_recovery_pct', 'perturb_failure_recovery_pct']].plot.bar(
    ax=axes[2], color=['#54A24B', '#E45756'])
axes[2].set(title='Recovery among stock failures', ylabel='Recovered failures (%)',
            xlabel='Root selection')
for axis in axes: axis.grid(axis='y', alpha=.25)
fig.tight_layout(); plt.show()

## Fresh-noise versus P&P-perturbation candidates

In [ ]:
family_table = (candidates[candidates.candidate_family.ne('stored_source')]
    .groupby(['candidate_family', 'selection_strategy'], observed=True)
    .agg(candidates=('candidate_id', 'size'), trees=('candidate_group_id', 'nunique'),
         branch_sr_pct=('success', lambda x: 100*x.mean()),
         mean_pnp_u10=('pnp_u10', 'mean'))
    .reset_index())
display(family_table)
slot_table = (candidates.groupby(['candidate_kind', 'candidate_family'], observed=True)
    .agg(candidates=('candidate_id', 'size'), branch_sr_pct=('success', lambda x: 100*x.mean()),
         mean_pnp_u10=('pnp_u10', 'mean')).reset_index())
display(slot_table)

stock_failures = complete[~complete.stock_success]
contingency = pd.crosstab(
    stock_failures.fresh_any_success.rename('fresh has success'),
    stock_failures.perturb_any_success.rename('P&P perturb has success'))
print(f'Stock-failure trees: {len(stock_failures)}')
display(contingency)
print({'fresh_only_recoveries': int((stock_failures.fresh_any_success & ~stock_failures.perturb_any_success).sum()),
       'perturb_only_recoveries': int((~stock_failures.fresh_any_success & stock_failures.perturb_any_success).sum()),
       'both_recover': int((stock_failures.fresh_any_success & stock_failures.perturb_any_success).sum()),
       'neither_recovers': int((~stock_failures.fresh_any_success & ~stock_failures.perturb_any_success).sum())})

## Does candidate P&P uncertainty rank branch outcomes?

In [ ]:
u_ranking = uncertainty_ranking_summary(candidates)
display(u_ranking)

counter = candidates[candidates.candidate_family.ne('stored_source')].dropna(subset=['pnp_u10'])
fig, ax = plt.subplots(figsize=(7, 4))
for success, label, color in [(True, 'successful branch', '#54A24B'),
                              (False, 'failed branch', '#E45756')]:
    values = counter.loc[counter.success.eq(success), 'pnp_u10']
    ax.hist(values, bins=30, density=True, alpha=.45, label=f'{label} (n={len(values)})',
            color=color)
ax.set(title='Candidate root U10 versus eventual branch outcome', xlabel='Candidate P&P U10',
       ylabel='Density'); ax.legend(); ax.grid(alpha=.2); plt.show()

## Root U10, suite breakdown, and training-relevant counts

In [ ]:
root_bins = complete.copy()
root_bins['u10_quintile'] = pd.qcut(root_bins.source_root_u10, 5, duplicates='drop')
u10_rows = []
for label, group in root_bins.groupby('u10_quintile', observed=True):
    stock_failures_in_bin = int((~group.stock_success).sum())
    u10_rows.append({
        'u10_quintile': label, 'trees': len(group),
        'mean_root_u10': group.source_root_u10.mean(),
        'stock_sr_pct': 100*group.stock_success.mean(),
        'mixed_pct': 100*group.mixed_outcomes.mean(),
        'stock_failures': stock_failures_in_bin,
        'failure_recovery_pct': (100*group.either_recovers_failure.sum()/stock_failures_in_bin
                                 if stock_failures_in_bin else np.nan),
        'counter_branch_sr_pct': 100*group.counterfactual_success_fraction.mean(),
    })
u10_table = pd.DataFrame(u10_rows)
display(u10_table)

suite_table = summarize_trees(trees, ['suite', 'selection_strategy'])
display(suite_table)
suite_plot = summarize_trees(trees, 'suite').set_index('suite')
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
suite_plot.mixed_trees_pct.plot.bar(ax=axes[0], color='#4C78A8')
axes[0].set(title='Mixed trees by suite', ylabel='Mixed trees (%)', xlabel='')
suite_plot.oracle_gain_vs_stock_pp.plot.bar(ax=axes[1], color='#F58518')
axes[1].axhline(0, color='black', lw=1)
axes[1].set(title='Oracle gain by suite', ylabel='Gain over stock (pp)', xlabel='')
for axis in axes: axis.grid(axis='y', alpha=.25)
fig.tight_layout(); plt.show()

display(complete.successful_counterfactuals.value_counts().sort_index().rename(
    'trees').to_frame().rename_axis('successful_counterfactuals_out_of_8'))

training_counts = {
    'complete_trees': len(complete),
    'candidate_rows': len(candidates),
    'mixed_trees': int(complete.mixed_outcomes.sum()),
    'counterfactual_mixed_trees': int(complete.counterfactual_mixed.sum()),
    'stock_failure_trees': int((~complete.stock_success).sum()),
    'stock_failures_with_any_successful_counterfactual': int(complete.either_recovers_failure.sum()),
    'success_failure_candidate_pairs': int(sum(
        g.success.sum() * (~g.success).sum()
        for _, g in candidates.groupby('candidate_group_id'))),
}
training_counts